<a href="https://colab.research.google.com/github/SrijanKumar123/flyrank-ml-internship/blob/main/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SrijanKumar123/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb
import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN '{token}'
    )
""")

In [ ]:
REL = """
read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

In [ ]:
%pip -q install -U duckdb huggingface_hub

import duckdb
from google.colab import userdata
from huggingface_hub import whoami

token = userdata.get("HF_TOKEN")

print("Token found:", token is not None)
print("Logged in as:", whoami(token=token)["name"])

Token found: True
Logged in as: srijan317


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Logisitic Regression is chosen because it provides direct linear coefficient weights, allowing us to see exactly how features like search position or impression volume impact prediction log-odds.  Logistic Regression serves as the direct, honest baseline model.  

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

dataset = con.sql(f""" SELECT client_hash_id, gsc_avg_position, gsc_impressions, ga4_pageviews, ga4_sessions, gsc_clicks
FROM {REL} """).df()
dataset["ctr"] = dataset["gsc_clicks"] / dataset["gsc_impressions"].replace(
    0, np.nan
)

dataset["is_striking_distance"] = (
    (dataset["gsc_avg_position"] > 3) &
     (dataset["gsc_avg_position"] <= 20) &
 (dataset["gsc_impressions"] >= 500)
 ).astype(int)

dataset["ctr"] = dataset["ctr"].fillna(0)
dataset["ga4_pageviews"] = dataset["ga4_pageviews"].fillna(0)
dataset["ga4_sessions"] = dataset["ga4_sessions"].fillna(0)
dataset["gsc_avg_position"] = dataset["gsc_avg_position"].fillna(100.0)

y = dataset["is_striking_distance"]
X = dataset[["gsc_avg_position", "gsc_impressions", "ctr",  "ga4_pageviews", "ga4_sessions"]]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Grouping by client domain guarantees that all performance records belonging to a specific client stay strictly within either the training set or the validation set.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupKFold

groups = dataset["client_hash_id"]
gkf = GroupKFold(n_splits=4)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

gkf = GroupKFold(n_splits=4)
groups = dataset['client_hash_id']

f1_scores, precision_scores, recall_scores = [], [], []

for train_idx, val_idx in gkf.split(X, y, groups=groups):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    model = LogisticRegression()
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_val_scaled)

    f1_scores.append(f1_score(y_val, y_pred, zero_division=0))
    precision_scores.append(precision_score(y_val, y_pred, zero_division=0))
    recall_scores.append(recall_score(y_val, y_pred, zero_division=0))

print(f"Mean Out-of-Fold F1-Score: {np.mean(f1_scores):.4f}")
print(f"Mean Out-of-Fold Precision: {np.mean(precision_scores):.4f}")
print(f"Mean Out-of-Fold Recall: {np.mean(recall_scores):.4f}")

Mean Out-of-Fold F1-Score: 0.4288
Mean Out-of-Fold Precision: 0.6498
Mean Out-of-Fold Recall: 0.3330


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

When Logistic Regression flags a content page as "Striking Distance," it is correct 65% of the time, providing reliable signals for content teams.

The model misses 67% of actual striking-distance opportunities. Because 99%+ of the dataset consists of zero-click/unranked pages, the linear model defaults toward conservative predictions to avoid false positives.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_weights = pd.Series(model.coef_[0], index=X.columns).sort_values(ascending=False)
print("Feature Coefficients:")
print(feature_weights)

Feature Coefficients:
gsc_impressions     0.758453
ga4_pageviews       0.308220
ctr                -0.028267
ga4_sessions       -0.268292
gsc_avg_position   -6.494522
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.